# 73 Job · Tarea 04 · Gold y Cierre

Esta es la tarea de *fan-in* y corre con `run_if: ALL_DONE`, incluso cuando se activa la rama de fallas. Por defecto construye el objeto Gold para BI; con `accion=registrar_falla` registra la rama negativa sin duplicar otro notebook.

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [ ]:
# Recibe el identificador de corrida y selecciona entre cierre normal o registro de falla.
dbutils.widgets.text("run_id", "manual", "ID de la corrida")
dbutils.widgets.text("accion", "cierre", "Acción")
run_id = dbutils.widgets.get("run_id")
accion = dbutils.widgets.get("accion").strip().lower()
assert accion in {"cierre", "registrar_falla"}, f"Acción no soportada: {accion}"

# Centraliza la lectura de task values y devuelve defaults durante una ejecución manual.
def obtener_task_value(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, default=default)
    except Exception:
        return default

# Recupera las métricas que se incluirán en la bitácora de la corrida.
filas_ingeridas = int(obtener_task_value("01_ingesta_bronze", "filas_ingeridas", -1))
filas_malas = int(obtener_task_value("02_control_calidad", "filas_malas", -1))
print(f"run_id={run_id}, accion={accion}, filas_ingeridas={filas_ingeridas}, filas_malas={filas_malas}")

In [ ]:
TABLA_METRICAS = f"{CATALOG}.{SCHEMA}.movielens_metricas_genero"
TABLA_GOLD = f"{CATALOG}.{SCHEMA}.movielens_gold_genero"

# Solo reconstruye Gold cuando esta es la acción de cierre y la calidad resultó aceptable.
if accion == "cierre" and filas_malas in {-1, 0}:
    assert spark.catalog.tableExists(TABLA_METRICAS), (
        f"No existe {TABLA_METRICAS}. Ejecutá primero el notebook 72 o la rama For each del job.")
    # Publica las métricas consolidadas como tabla gobernada para consumo analítico.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TABLA_GOLD}
    COMMENT 'Métricas Gold de ratings de MovieLens por género para consumo analítico y BI'
    AS SELECT genero, cantidad_ratings, rating_promedio, desviacion_estandar,
              peliculas_distintas, actualizado_ts
       FROM {TABLA_METRICAS}
    """)
    # Documenta cada columna para que su significado aparezca en Catalog Explorer.
    comentarios = {
        "genero": "Género cinematográfico de MovieLens",
        "cantidad_ratings": "Cantidad de calificaciones válidas",
        "rating_promedio": "Promedio de calificación en escala de 0.5 a 5.0",
        "desviacion_estandar": "Desviación estándar muestral de las calificaciones",
        "peliculas_distintas": "Cantidad de películas distintas calificadas",
        "actualizado_ts": "Marca temporal de actualización de la métrica"
    }
    for columna, comentario in comentarios.items():
        spark.sql(f"ALTER TABLE {TABLA_GOLD} ALTER COLUMN {columna} COMMENT '{comentario}'")
    # Expone una vista ordenada y más simple para herramientas de BI.
    spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.v_top_generos
    COMMENT 'Vista BI de géneros ordenados por rating promedio y volumen'
    AS SELECT genero, cantidad_ratings, rating_promedio, desviacion_estandar, peliculas_distintas
       FROM {TABLA_GOLD}
       ORDER BY rating_promedio DESC, cantidad_ratings DESC
    """)
    display(spark.table(f"{CATALOG}.{SCHEMA}.v_top_generos"))
elif accion == "cierre":
    # Una corrida rechazada no debe reemplazar la última versión Gold válida.
    print("La calidad no pasó: se conserva la última Gold válida y se registra el cierre fallido.")
else:
    print("Rama false confirmada: se registrará el fallo de calidad.")

Una **view** guarda una consulta y calcula el resultado al leerla; una **materialized view** mantiene resultados precomputados y actualizados por Databricks; una **streaming table** procesa entradas incrementales con semántica de streaming. Para este conjunto pequeño y una consulta directa de BI, la view evita almacenamiento y mantenimiento adicionales.

In [ ]:
from pyspark.sql import Row
from pyspark.sql import functions as F

# Traduce la acción y la métrica de calidad a un estado legible para operación.
if accion == "registrar_falla":
    estado = "CALIDAD_FALLIDA"
elif filas_malas > 0:
    estado = "FINALIZADO_CON_RECHAZOS"
else:
    estado = "COMPLETADO"

# Construye un registro con esquema estable y la marca temporal del cierre.
registro = spark.createDataFrame([Row(
    run_id=str(run_id),
    filas_ingeridas=int(filas_ingeridas),
    filas_malas=int(filas_malas),
    estado=estado
)]).withColumn("ts", F.current_timestamp()).select(
    "run_id", "ts", "filas_ingeridas", "filas_malas", "estado")

# Agrega la evidencia a la bitácora sin borrar corridas anteriores.
(registro.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.bitacora_corridas"))
print(f"Corrida {run_id} registrada con estado {estado}.")
display(registro)

## Cierre

- Integraste ramas del DAG mediante una tarea `ALL_DONE`.
- Construiste una tabla Gold documentada y una view para BI cuando la calidad lo permitió.
- Diferenciaste view, materialized view y streaming table.
- Registraste el resultado de la corrida y sus principales *task values*.